# Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU — Qdrant Production Demo

## Tổng quan / Overview

**VI:** Notebook công khai này trình diễn end-to-end workflow đã được qualify cho **WeMM-Embedding-9B**, model do **đội ngũ WeChat Vision tại Tencent** phát triển, trên Kaggle T4×2, kết hợp embedding đa phương thức với **Qdrant** để kiểm tra semantic retrieval song ngữ/cross-modal và visual robustness retrieval.

**EN:** This public notebook demonstrates the qualified end-to-end workflow for **WeMM-Embedding-9B**, developed by the **WeChat Vision Team at Tencent**, on Kaggle T4×2, combining multimodal embeddings with **Qdrant** for bilingual/cross-modal semantic retrieval and visual robustness retrieval.

Notebook giữ nguyên frozen science/runtime authority; phần presentation chỉ tổ chức lại cách người dùng đọc và chạy demo.

### Notebook gồm những phần nào? / What does the notebook cover?

1. **Steps 1–5 — Bootstrap + system setup**
   - chuẩn bị public release source;
   - kiểm tra Kaggle T4×2 và attached Inputs;
   - chuẩn bị Qdrant;
   - load model worker và xác minh dual-GPU placement.
2. **Step 6 — Bilingual text retrieval**
   - English ↔ Vietnamese semantic retrieval.
3. **Step 7A — Semantic image→text retrieval**
   - image queries tìm entity text trong production semantic corpus.
4. **Step 7B — Visual robustness retrieval**
   - transformed-image queries tìm lại original image trong temporary four-image gallery.
5. **Step 8 — Closeout + acceptance**
   - giải phóng GPU, đóng Qdrant sạch và in final scorecard.

### Hai evaluation spaces phải được đọc riêng / Keep the two evaluation spaces separate

- **Semantic corpus retrieval:** Step 6 + Step 7A, search space **99,967 entities per production collection**, tổng hợp **36/36 TOP-1** trên các frozen public paths.
- **Visual robustness retrieval:** Step 7B, search space là **temporary gallery gồm 4 original images**, tổng hợp **32/32 TOP-1** với raw cosine threshold `0.90`.
- **68/68 PASS** chỉ là tổng số retrieval checks đã thực thi trong hai search spaces khác nhau; không phải “68/68 accuracy” hay một universal benchmark score.

### Cách chạy / Execution model

Notebook có **5 executable code cells** và nên được chạy từ trên xuống dưới trong một **fresh kernel → single Run All → no repair/rerun** khi thu thập publication evidence.

Markdown cell kế tiếp chứa toàn bộ yêu cầu Kaggle và hướng dẫn attach Model/Dataset trước khi Code Cell 1 bắt đầu bootstrap.


## Steps 1/8–>5/8 — Bootstrap + chuẩn bị hệ thống / Bootstrap + system setup

### Trước khi chạy / Before you run

**VI:** Notebook này được thiết kế cho một **fresh Kaggle Notebook** với đúng cấu hình đã được qualify. Trước khi chạy code cell bên dưới, hãy hoàn tất các bước sau trong Kaggle:

1. Mở **Notebook options / Settings** và chọn **GPU T4 ×2**.
2. Bật **Internet = ON**.
3. Trong panel **Input**, chọn **Add Input** và attach **Model**:
   `dangkhoa2016/tencent-wemm-embedding-9b` — version `1`.
4. Tiếp tục **Add Input** và attach **Dataset**:
   `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots` — version `1`.
5. Xác nhận cả Model và Dataset đã xuất hiện dưới Kaggle Input trước khi bấm **Run All**.

**EN:** This notebook is intended for a **fresh Kaggle Notebook** using the qualified configuration. Before running the code cell below:

1. Open **Notebook options / Settings** and select **GPU T4 ×2**.
2. Set **Internet = ON**.
3. In the **Input** panel, choose **Add Input** and attach the **Model**:
   `dangkhoa2016/tencent-wemm-embedding-9b` — version `1`.
4. Use **Add Input** again and attach the **Dataset**:
   `dangkhoa2016/wemm-embedding-9b-v1-qdrant-snapshots` — version `1`.
5. Confirm both inputs are attached before clicking **Run All**.

> Kaggle mounts attached read-only inputs below `/kaggle/input`. The public integration validates that required model/data files resolve from the Kaggle Input boundary; it does not silently substitute unrelated local files.

### Cell code bên dưới làm gì / What the next code cell does

The first code cell performs the complete setup phase before any retrieval example is executed:

- checks out **presentation source from public release `v1.0.0`**;
- bootstraps the frozen science/runtime authority at
  `224f07cd1d6eb174d3532c9eaeeb9abd606a857f`;
- installs `requirements-kaggle.txt` + `requirements-demo.txt` without replacing the qualified CUDA/PyTorch runtime;
- validates the T4×2 environment and required Kaggle Inputs;
- reuses an already-clean Qdrant working copy when valid, otherwise restores from the verified read-only snapshot source;
- starts the model worker and validates dual-T4 placement before continuing.

The workflow is **fail-closed**: missing inputs, unsupported accelerator, invalid snapshot state, or invalid model placement should stop execution rather than silently falling back.

### Kết quả cần thấy / Expected evidence

Look for setup markers including:

- `PUBLIC_NOTEBOOK_PRESENTATION_SOURCE=PASS`
- `PUBLIC_NOTEBOOK_PRESENTATION_REF=v1.0.0`
- `PUBLIC_DEMO_SOURCE_BOOTSTRAP=PASS`
- `RUNTIME_SOURCE_COMMIT=224f07cd1d6eb174d3532c9eaeeb9abd606a857f`
- `NOTEBOOK_PHASE_SETUP=PASS`

Only continue when this setup cell completes successfully. Qdrant and the GPU worker remain live for Steps 6, 7A and 7B.

In [ ]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys

REPO = "https://github.com/dangkhoa2016/Tencent-WeMM-Embedding-9B-Kaggle-T4x2-GPU.git"
PUBLIC_RELEASE_REF = "v1.0.0"
SOURCE_ROOT = Path("/kaggle/working/wemm-public-notebook-source-v1.0.0")

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
subprocess.run(["git", "init", "-q"], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "remote", "add", "origin", REPO], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", PUBLIC_RELEASE_REF], cwd=SOURCE_ROOT, check=True)
subprocess.run(["git", "checkout", "-q", "--detach", "FETCH_HEAD"], cwd=SOURCE_ROOT, check=True)

sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
import wemm_notebook
from wemm_notebook import start_public_session, run_step6, run_step7a, run_step7b, run_step8

presentation_file = Path(wemm_notebook.__file__).resolve()
assert SOURCE_ROOT.resolve() in presentation_file.parents, (presentation_file, SOURCE_ROOT)
print("PUBLIC_NOTEBOOK_PRESENTATION_SOURCE=PASS", flush=True)
print("PUBLIC_NOTEBOOK_PRESENTATION_REF=" + PUBLIC_RELEASE_REF, flush=True)

demo = start_public_session()

## Step 6/8 — Truy xuất văn bản song ngữ / Bilingual text retrieval

### Mục tiêu / Goal

**VI:** Phase này kiểm tra semantic retrieval chéo ngôn ngữ trên production Qdrant corpus. Showcase được freeze gồm **5 cặp EN↔VI** và **20 retrieval paths**: English→Vietnamese và Vietnamese→English ở cả `4096d` và `1024d`.

**EN:** This phase evaluates cross-language semantic retrieval over the production Qdrant corpus. The frozen showcase contains **5 EN↔VI examples** and **20 retrieval paths**: English→Vietnamese and Vietnamese→English at both `4096d` and `1024d`.

### Cell code bên dưới làm gì / What the next code cell does

`run_step6(demo)` reuses the already-loaded GPU worker and Qdrant service from the setup phase. For each frozen example it embeds the full query, searches the corresponding named vector space, and prints human-readable retrieval evidence.

The presentation intentionally shows:

- **full query text**;
- **full candidate text**;
- **TOP-1 WINNER / KẾT QUẢ #1**;
- **NEAREST COMPETITOR / ĐỐI THỦ GẦN NHẤT**;
- raw cosine similarity for comparison.

No text is clipped by the presentation layer. Raw cosine is a **similarity score**, not a confidence percentage.

### Cách đọc kết quả / How to read the output

A passing path means the expected bilingual entity is ranked **#1** for that defined query/dimension path. This is a reproducibility result for the frozen showcase and corpus configuration, not a claim of universal model accuracy.

Expected phase markers include:

- `FROZEN_BILINGUAL_SHOWCASE=PASS`
- `TEXT_SHOWCASE_ALL_TOP1=5/5`
- `TEXT_SHOWCASE_STRICT_ALL_TOP3=PASS`
- `STEP_6_PRESENTATION_LAYER=HUMAN_FIRST_FULL_TEXT`
- `NOTEBOOK_PHASE_STEP6=PASS`

Do not run this cell before the setup cell; the sectioned runner enforces phase order.

In [ ]:
text_results = run_step6(demo)

## Step 7A/8 — Truy xuất semantic ảnh→văn bản / Semantic image→text retrieval

### Mục tiêu và search space / Goal and search space

**VI:** Step 7A kiểm tra khả năng dùng **ảnh làm query** để truy xuất entity text song ngữ trong **production Qdrant corpus có 99,967 entities mỗi collection**. Frozen showcase gồm **4 image→text examples**.

**EN:** Step 7A uses an **image as the query** to retrieve bilingual text entities from the **production Qdrant corpus containing 99,967 entities per collection**. The frozen showcase contains **4 image→text examples**.

Each image is evaluated across:

- English candidate space;
- Vietnamese candidate space;
- `4096d`;
- `1024d`.

That gives **4 images × 2 languages × 2 dimensions = 16 semantic retrieval paths**.

### Cell code bên dưới làm gì / What the next code cell does

`run_step7a(demo)` embeds each frozen image with the same already-loaded WeMM worker, searches the production semantic collections, and reports the retrieved entity/text plus raw cosine similarity.

This phase **does not create the four-image robustness gallery used in Step 7B**. Step 7A and Step 7B have different search spaces and must be interpreted separately.

### Cách đọc kết quả / How to read the output

For every defined path, the expected entity should appear at rank **#1**. Raw cosine remains an unscaled similarity value.

Look for the four frozen examples, their English/Vietnamese results at both dimensions, and:

- `NOTEBOOK_PHASE_STEP7A=PASS`

The combined semantic result from Step 6 + Step 7A is summarized later in Step 8 as **36/36 TOP-1** over the 99,967-entity corpus.

In [ ]:
image_results = run_step7a(demo)

## Step 7B/8 — Độ bền truy xuất hình ảnh / Visual robustness retrieval

### Mục tiêu / Goal

**VI:** Step 7B là một evaluation **khác search space hoàn toàn** với Step 6/7A. Nó kiểm tra transformed-image→original-image retrieval trong một **temporary curated gallery chỉ gồm đúng 4 ảnh gốc**.

**EN:** Step 7B uses a **different search space** from Steps 6/7A. It evaluates transformed-image→original-image retrieval in a **temporary curated gallery containing exactly four originals**.

Frozen visual entities:

- `Q19217` — Carrie Lam
- `Q10489198` — Ho Chi Minh University of Education
- `Q168751` — Duke University
- `Q51756` — China Airlines

Each original is queried through four real transforms:

- resize to `80%`;
- JPEG quality `90`;
- center crop to `96%`;
- brightness `103%`.

Both `4096d` and `1024d` are evaluated, giving:

`4 entities × 4 transforms × 2 dimensions = 32 retrieval paths`.

### Cell code bên dưới làm gì / What the next code cell does

`run_step7b(demo)` creates an isolated local Qdrant gallery for the four originals, embeds transformed queries, searches both dimensions, validates rank and threshold, then deletes the temporary Qdrant instance when finished.

PASS requires **every one of the 32 paths** to:

- retrieve the correct original at **rank #1**;
- achieve **raw cosine >= 0.90**.

Strict invariants:

- no cosine rescaling;
- no threshold relaxation;
- no mutation of the production semantic collections;
- temporary visual Qdrant must be deleted.

### Kết quả cần thấy / Expected evidence

Expected markers include:

- `HIGH_CONFIDENCE_VISUAL_RETRIEVAL=PASS`
- `VISUAL_RETRIEVAL_EXAMPLES=4`
- `VISUAL_RETRIEVAL_PATHS_TOP1=32/32`
- `VISUAL_RETRIEVAL_THRESHOLD=0.90`
- `VISUAL_RETRIEVAL_RAW_COSINE_RESCALED=NO`
- `VISUAL_RETRIEVAL_THRESHOLD_RELAXED=NO`
- `VISUAL_RETRIEVAL_PRODUCTION_COLLECTIONS_MUTATED=NO`
- `VISUAL_RETRIEVAL_TEMP_QDRANT=DELETED`
- `NOTEBOOK_PHASE_STEP7B=PASS`

These 32 checks must **not** be merged conceptually with the 99,967-entity semantic search space.

In [ ]:
visual_results = run_step7b(demo)

## Step 8/8 — Đóng phiên + nghiệm thu / Closeout + acceptance

### Cell code bên dưới làm gì / What the next code cell does

`run_step8(demo, visual_results)` closes the live demo session only after Steps 6, 7A and 7B have completed in order. It:

- releases the GPU worker;
- verifies GPU/VRAM reclaim;
- stops Qdrant cleanly;
- seals the production Qdrant storage state;
- confirms that no persistent Qdrant snapshot copy was created by the demo;
- prints the final public scorecard and sectioned-runner acceptance markers.

### Cách đọc scorecard / How to read the scorecard

The final report intentionally keeps the two evaluation spaces separate:

- **Semantic corpus retrieval: 36/36 TOP-1**
  - Step 6 + Step 7A
  - production corpus: **99,967 entities per collection**
- **Visual robustness retrieval: 32/32 TOP-1**
  - Step 7B
  - temporary gallery: **4 original images**
- **Total executed retrieval checks: 68/68 PASS**

`68/68` means only **TOTAL EXECUTED RETRIEVAL CHECKS**. It is **not** “68/68 accuracy”, “100% accuracy”, or a universal benchmark score.

### Final acceptance evidence

A clean final run should include:

- `WORKER_LIFECYCLE_GPU_RECLAIM=PASS`
- `QDRANT_STORAGE_SEAL=PASS`
- `QDRANT_SNAPSHOT_PERSISTENT_COPY=NO`
- `PUBLIC_DEMO_EXTENDED_SCORECARD=PASS`
- `SEMANTIC_CORPUS_RETRIEVAL_PATHS_TOP1=36/36`
- `SEMANTIC_CORPUS_SEARCH_SPACE_ENTITIES=99967`
- `VISUAL_ROBUSTNESS_RETRIEVAL_PATHS_TOP1=32/32`
- `VISUAL_ROBUSTNESS_SEARCH_SPACE_IMAGES=4`
- `PUBLIC_DEMO_TOTAL_EXECUTED_RETRIEVAL_CHECKS=68/68`
- `NOTEBOOK_PHASE_STEP8=PASS`
- `SECTIONED_NOTEBOOK_RUNNER=PASS`
- `NOTEBOOK_EXECUTABLE_CELLS=5`
- `STEP_7A_7B_SEPARATE_CELLS=PASS`

For publication evidence, the preferred run is a **fresh kernel → single Run All → no repair/rerun**, with all five code cells receiving execution counts. If Kaggle stops between cells without a Python traceback and the next cell remains unexecuted, preserve that evidence instead of silently presenting a repaired run as a clean single pass.

In [ ]:
final_summary = run_step8(demo, visual_results)